# Multimodel AI - Blood Work Analysis

<p>Use a vision-capable LLM to read and interpret it.</P>

In [9]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

## Encode the image and send to the vision model

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAccAAAJCCAYAAAC1aQxEAAAAL3RFWHRDcmVhdGlvbiBUaW1lAFN1biAwMiBBdWcgMjAyNiAxMDowNzo0NiBBTSArMDUzMKqpAG4AAAAZdEVYdFNvZnR3YXJlAGdub21lLXNjcmVlbnNob3TvA78+AAEZ7ElEQVR4nOz9D1STZ77vDX/P'

In [5]:
llm = ChatGroq(model="qwen/qwen3.6-27b")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text", "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)


<think>
The user wants me to extract data from a medical report image.
I need to identify the different sections: CBC, Lipid Panel, Metabolic Panel, and Liver Function.
For each section, I need to list the test name, the result value, and the normal range provided.
Finally, I need to compare the result to the normal range and flag if it's abnormal.

**Section 1: COMPLETE BLOOD COUNT (CBC)**
*   Hemoglobin: 15.1 g/dL. Normal: 13.5-17.5. (15.1 is within range).
*   Hematocrit: 44%. Normal: 41-53%. (44 is within range).
*   WBC: 6.8 x10^3/uL. Normal: 4.5-11.0. (6.8 is within range).
*   Platelets: 220 x10^3/uL. Normal: 150-400. (220 is within range).

**Section 2: LIPID PANEL**
*   Total Cholesterol: 238 mg/dL. Normal: <200. (238 > 200, so HIGH).
*   LDL Cholesterol: 162 mg/dL. Normal: <100. (162 > 100, so HIGH).
*   HDL Cholesterol: 36 mg/dL. Normal: >40. (36 < 40, so LOW).
*   Triglycerides: 188 mg/dL. Normal: <150. (188 > 150, so HIGH).

**Section 3: METABOLIC PANEL**
*   Glucose (Fas

In [12]:
DIET_PLAN = {
    "high_cholesterol": {
        "eat": [
            "vegetables", "fruits", "oats", "barley", "beans", "lentils",
            "whole grains", "nuts", "seeds", "fish", "olive oil"
        ],
        "limit_or_avoid": [
            "processed meats", "fried foods", "fast food",
            "butter", "ghee", "full-fat dairy", "trans fats",
            "sugary drinks", "refined carbohydrates"
        ]
    },

    "cbc_low_hemoglobin_or_hematocrit": {
        "eat": [
            "lean meat", "fish", "eggs", "lentils", "beans",
            "dark leafy greens", "iron-fortified cereals",
            "vitamin C foods such as oranges, guava, and bell peppers"
        ],
        "limit_or_avoid": [
            "tea or coffee with iron-rich meals",
            "self-prescribed iron supplements without medical advice"
        ]
    },

    "cbc_high_wbc": {
        "eat": [
            "adequate protein", "vegetables", "fruits",
            "whole grains", "water and other unsweetened fluids"
        ],
        "limit_or_avoid": [
            "excess alcohol", "highly processed foods"
        ],
        "note": "A high WBC count needs clinical assessment for causes such as infection or inflammation; food alone does not treat it."
    },

    "cbc_low_platelets": {
        "eat": [
            "balanced meals with fruits, vegetables, whole grains",
            "lean protein", "adequate fluids"
        ],
        "limit_or_avoid": [
            "alcohol",
            "unprescribed supplements or medicines that may affect bleeding"
        ],
        "note": "Low platelets should be assessed by a clinician, especially if there is easy bruising or bleeding."
    },

    "high_fasting_glucose_or_hba1c": {
        "eat": [
            "non-starchy vegetables", "beans", "lentils",
            "whole grains in moderate portions", "nuts",
            "lean proteins", "plain yogurt"
        ],
        "limit_or_avoid": [
            "sugary drinks", "sweets", "fruit juice",
            "white bread", "white rice in large portions",
            "refined snacks"
        ]
    },

    "abnormal_kidney_function": {
        "eat": [
            "a kidney-specific plan set with a doctor or dietitian",
            "fresh, minimally processed foods"
        ],
        "limit_or_avoid": [
            "high-sodium packaged foods",
            "excess protein supplements",
            "salt substitutes containing potassium unless approved"
        ],
        "note": "Diet changes for reduced eGFR or high creatinine must be individualized because potassium, phosphorus, protein, and fluid needs can differ."
    },

    "elevated_liver_enzymes": {
        "eat": [
            "vegetables", "fruits", "whole grains",
            "beans", "fish", "lean protein", "unsweetened coffee if tolerated"
        ],
        "limit_or_avoid": [
            "alcohol", "sugary drinks", "excess fructose",
            "fried foods", "processed meats",
            "unnecessary herbal or bodybuilding supplements"
        ],
        "note": "Persistent ALT or AST elevation should be reviewed by a clinician to identify the cause."
    },

    "high_bilirubin": {
        "eat": [
            "regular balanced meals", "adequate fluids",
            "fruits", "vegetables", "lean proteins"
        ],
        "limit_or_avoid": [
            "alcohol", "fasting or crash diets",
            "unreviewed supplements"
        ],
        "note": "High bilirubin requires medical evaluation, particularly with yellow eyes/skin, dark urine, pale stools, fever, or abdominal pain."
    }
}

@tool
def get_diet_plan(name: str) -> str:
    """Retrieves a diet plan by condition key.
    Valid keys: 'high_cholesterol', 'cbc_low_hemoglobin_or_hematocrit', 
    'cbc_high_wbc', 'cbc_low_platelets', 'high_fasting_glucose_or_hba1c', 
    'abnormal_kidney_function', 'elevated_liver_enzymes', 'high_bilirubin'."""
    dp = DIET_PLAN.get(name)
    if not dp:
        return f"No specific diet plan found."
    return str(dp)

llm = ChatGroq(model="qwen/qwen3.6-27b", temperature=0)

agent = create_agent(
    llm,
    tools = [get_diet_plan],
    system_prompt = (
    "You are a helpful medical assistant that provides general diet-plan "
    "suggestions based on health metrics. Do not diagnose medical conditions. "
    "Encourage users to consult a qualified healthcare professional for "
    "abnormal, severe, or persistent results."
    ))

def ask(question: str):
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=[
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                        {"type": "text", "text": question},
                    ]
                )
            ]
        }
    )
    print(result["messages"][-1].content)

In [13]:
ask("This is a blood work report. Extract all test results and flag any values outside the normal range.")

Based on the blood work report provided, here are the extracted test results.

### **Test Results Summary**

**Complete Blood Count (CBC)**
*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5–17.5) — *Normal*
*   **Hematocrit:** 44% (Normal: 41–53%) — *Normal*
*   **WBC:** 6.8 x10^3/uL (Normal: 4.5–11.0) — *Normal*
*   **Platelets:** 220 x10^3/uL (Normal: 150–400) — *Normal*

**Lipid Panel**
*   **Total Cholesterol:** 238 mg/dL (Normal: <200) — **High**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) — **High**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) — **Low**
*   **Triglycerides:** 188 mg/dL (Normal: <150) — **High**

**Metabolic Panel**
*   **Glucose (Fasting):** 92 mg/dL (Normal: 70–99) — *Normal*
*   **HbA1c:** 5.3% (Normal: <5.7%) — *Normal*
*   **Creatinine:** 1.0 mg/dL (Normal: 0.7–1.3) — *Normal*
*   **eGFR:** 82 mL/min (Normal: >60) — *Normal*

**Liver Function**
*   **ALT:** 28 U/L (Normal: 7–40) — *Normal*
*   **AST:** 25 U/L (Normal: 10–40) — *Normal*
*   **Bilirubin Tota